Item Classification - Volume & Gross Profit
Computes per-item volume and gross profit with precomputed percentile ranks. Thresholds are applied as filters, not recomputed.

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../../_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_silver, save_gold
from src.analysis.item_classification import classify_items, summarize_warranty_claims, filter_by_percentile
from src.analysis.item_velocity import compute_item_velocity
import pandas as pd
import io

blob_service = get_blob_service(storage_account_name, storage_account_key)

ANALYSIS_BASE = "live/battery/analysis"

Load analysis silver

In [0]:
analysis_silver  = read_silver(blob_service, f"{ANALYSIS_BASE}/battery_analysis_clean_live.json")
analysis_silver ["posting_date"] = pd.to_datetime(analysis_silver ["posting_date"])
print(f"Analysis silver: {analysis_silver.shape}")

Classify items

In [0]:
item_classification = classify_items(analysis_silver)
print(item_classification.shape)
print(item_classification[["item_no", "total_units", "volume_percentile", "total_profit", "gp_percentile"]].head(10))

Save both formats

In [0]:
save_gold(blob_service, item_classification,  f"{ANALYSIS_BASE}/item_classification.parquet")

buffer = io.BytesIO()
item_classification.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)
blob_client = blob_service.get_blob_client(container="gold", blob= f"{ANALYSIS_BASE}/item_classification.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved item_classification.parquet and .xlsx")

Demo: apply a threshold as a pure filter

In [0]:
dbutils.widgets.text("volume_percentile_threshold", "80")
dbutils.widgets.text("gp_percentile_threshold", "80")

volume_threshold = float(dbutils.widgets.get("volume_percentile_threshold"))
gp_threshold = float(dbutils.widgets.get("gp_percentile_threshold"))

high_volume_items = filter_by_percentile(item_classification, "volume", volume_threshold)
high_gp_items = filter_by_percentile(item_classification, "gp", gp_threshold)

print(f"High volume items (>= {volume_threshold}th percentile): {len(high_volume_items)}")
print(f"High GP items (>= {gp_threshold}th percentile): {len(high_gp_items)}")

high_volume_low_gp = item_classification[
    (item_classification["volume_percentile"] >= volume_threshold) &
    (item_classification["gp_percentile"] < gp_threshold)
]
print(f"\nHigh volume but not high GP ({len(high_volume_low_gp)}): worth reviewing pricing")
print(high_volume_low_gp[["item_no", "total_units", "total_profit"]].head(10))

Warranty claims summary

In [0]:
warranty_summary = summarize_warranty_claims(analysis_silver)
print(warranty_summary.head(15))

Save warranty claims summary

In [0]:
save_gold(blob_service, warranty_summary, f"{ANALYSIS_BASE}/warranty_claims_summary.parquet")

buffer = io.BytesIO()
warranty_summary.to_excel(buffer, index=False, engine="openpyxl")
buffer.seek(0)
blob_client = blob_service.get_blob_client(container="gold", blob=f"{ANALYSIS_BASE}/warranty_claims_summary.xlsx")
blob_client.upload_blob(buffer, overwrite=True)
print("Saved warranty_claims_summary")